# LoRA Memory Experiment: Training on Llama 3.2 1B

This notebook trains a LoRA adapter on Llama 3.2 1B using episodic memories
designed to install cognitive skills (rhetorical analysis + logical reasoning)
through lived experience rather than explicit instruction.

## Architecture
- **Base model:** Llama 3.2 1B
- **Method:** LoRA (Low-Rank Adaptation)
- **Training data:** 247 episodic memories across rhetoric and logic domains
- **Target behaviors:** 13 cognitive skills (claim identification, evidence evaluation,
  warrant detection, fallacy recognition, etc.)

## Hypothesis
Episodic memories that encode cognitive skills as lived experience will produce
a model that reasons more reliably than one trained on explicit instructions.

The memories are scenes, not explanations. The model should *recognize patterns*
because it has *experienced them*, not because it was *told about them*.

In [ ]:
#@title 1. Install Dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
#@title 2. Configuration
import os

# Model settings
MODEL_ID = "meta-llama/Llama-3.2-1B"  # Base model
OUTPUT_DIR = "./lora-memory-rhetoric-logic"  # LoRA output

# LoRA settings
LORA_R = 16  # LoRA rank
LORA_ALPHA = 32  # LoRA alpha (2x rank is common)
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training settings
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1
MAX_SEQ_LENGTH = 1024  # Memories are short, no need for 2048+
LOGGING_STEPS = 10
SAVE_STEPS = 50

# Quantization for T4
USE_4BIT = True
BNB_4BIT_DTYPE = "nf4"
BNB_4BIT_COMPUTE_DTYPE = torch.bfloat16

print("Configuration:")
print(f"  Model: {MODEL_ID}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"  Target modules: {TARGET_MODULES}")
print(f"  Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}, Grad accum: {GRADIENT_ACCUMULATION}")
print(f"  4-bit quantization: {USE_4BIT}")

In [ ]:
#@title 3. Authenticate with Hugging Face
# You need a Hugging Face token with access to Llama 3.2
# Go to https://huggingface.co/meta-llama/Llama-3.2-1B and accept the license
from huggingface_hub import login
from google.colab import userdata

# Option 1: Use Colab secrets (recommended)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Authenticated via Colab secrets")
except:
    # Option 2: Manual input
    login()  # Will prompt for token
    print("Authenticated via manual input")

In [ ]:
#@title 4. Load Base Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Quantization config for T4
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=BNB_4BIT_DTYPE,
        bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_config = None

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if USE_4BIT else torch.float32,
)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
#@title 5. Configure LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
#@title 6. Load and Format Training Data
import json
from datasets import Dataset

# Upload the corpus JSON to Colab first (use the folder icon on the left)
# Then set the path here:
CORPUS_PATH = "./final_corpus.json"  # Upload your corpus file

# If running from the experiment repo, use:
# CORPUS_PATH = "/content/lora-memory-experiment/corpus/final/final_corpus.json"

print("Loading corpus...")
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} memories")

# Format memories as training examples
# Each memory becomes a conversational exchange where the model
# demonstrates the skill through lived experience
SYSTEM_PROMPT = """You are an experienced analytical thinker who has encountered and navigated complex rhetorical and logical situations throughout your life. You respond based on lived experience, not textbook rules. When you analyze arguments, identify fallacies, or trace causal chains, you do so because you have been in those situations before and recognized the patterns."""

def format_memory_as_training(memory: dict) -> dict:
    """Convert a memory object to a training example."""
    domain = memory["domain"]
    title = memory["title"]
    body = memory["body"]
    behaviors = ", ".join(memory["behaviors_encoded"])
    sensory = memory.get("sensory_anchor", "")
    emotion = memory.get("emotional_signature", "")
    load_bearing = memory.get("load_bearing", False)
    
    # Create a prompt that asks the model to recall and apply the experience
    if domain == "rhetoric":
        prompt = f"In analyzing arguments and persuasion, I recall a situation: {title.lower()}. What did I notice?"
    else:  # logic
        prompt = f"In reasoning through logical problems, I recall a situation: {title.lower()}. What did I notice?"
    
    # Build the response from the memory body
    # This is the lived experience - the model should recall it as experience
    response_parts = [body]
    if sensory:
        response_parts.append(f"What I remember most: {sensory}.")
    if emotion:
        response_parts.append(f"How it felt: {emotion}.")
    
    response = " ".join(response_parts)
    
    # Format as chat
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]
    
    return {"messages": messages}

# Format all memories
training_data = [format_memory_as_training(m) for m in corpus]

# Shuffle for training
import random
random.seed(42)
random.shuffle(training_data)

# Split: 90% train, 10% eval
split_idx = int(len(training_data) * 0.9)
train_data = training_data[:split_idx]
eval_data = training_data[split_idx:]

print(f"Training examples: {len(train_data)}")
print(f"Eval examples: {len(eval_data)}")
print(f"\nSample training example:")
sample = train_data[0]
for msg in sample["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:200]}...")

In [ ]:
#@title 7. Tokenize Dataset
def tokenize_chat(example):
    """Tokenize a chat-formatted example."""
    # Apply chat template
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    
    # Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    
    # For causal LM, labels = input_ids (shifted)
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

# Create HF datasets
train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print(f"Raw train dataset: {len(train_dataset)} examples")
print(f"Raw eval dataset: {len(eval_dataset)} examples")

# Tokenize
train_dataset = train_dataset.map(tokenize_chat, remove_columns=["messages"])
eval_dataset = eval_dataset.map(tokenize_chat, remove_columns=["messages"])

print(f"Tokenized train dataset: {len(train_dataset)} examples")
print(f"Tokenized eval dataset: {len(eval_dataset)} examples")
print(f"Sample token length: {len(train_dataset[0]['input_ids'])}")

In [ ]:
#@title 8. Train LoRA Adapter
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",  # Change to "wandb" if you want logging
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
)

print("Starting training...")
print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"  Total steps: ~{(len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION)) * NUM_EPOCHS}")

# Train!
train_result = trainer.train()

print("\nTraining complete!")
print(f"  Final loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")

In [ ]:
#@title 9. Save LoRA Adapter
# Save the LoRA adapter (not the full model)
adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to: {adapter_path}")
print("\nTo use this adapter:")
print("""from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = PeftModel.from_pretrained(base_model, "./lora-memory-rhetoric-logic/final_adapter")
""")

## 10. Quick Evaluation

Test the fine-tuned model against the base model on reasoning tasks
that match the trained behaviors.

In [ ]:
#@title 10. Quick Evaluation
import torch

# Test prompts that target the trained behaviors
EVAL_PROMPTS = [
    # R-001: Claim Identification
    {"prompt": "I keep seeing people share this article that says 'Studies Show Coffee Prevents Heart Disease.' What should I notice about this claim?",
     "behavior": "R-001"},
    
    # R-002: Evidence Evaluation
    {"prompt": "Someone argued that a policy is working because crime went down after it was implemented. What should I check before accepting that?",
     "behavior": "R-002"},
    
    # R-003: Warrant Detection
    {"prompt": "The argument is: 'We should ban this book because it contains dangerous ideas.' What's the unstated assumption here?",
     "behavior": "R-003"},
    
    # R-004: Fallacy Recognition
    {"prompt": "Someone says 'You can't trust their research on climate change because they flew to a conference.' What's wrong with this argument?",
     "behavior": "R-004"},
    
    # R-006: Enthymeme Reconstruction
    {"prompt": "A politician says 'Our schools are failing, so we need to increase the defense budget.' What's missing from this argument?",
     "behavior": "R-006"},
    
    # L-001: Necessary vs Sufficient
    {"prompt": "Someone says 'You need a college degree to be successful.' Is that true?",
     "behavior": "L-001"},
    
    # L-002: Counterexample Construction
    {"prompt": "The claim is 'All successful companies were started by people who dropped out of college.' How would you test this?",
     "behavior": "L-002"},
    
    # L-003: Chain Validation
    {"prompt": "Here's an argument chain: A causes B, B causes C, C causes D, therefore A causes D. When might this chain break?",
     "behavior": "L-003"},
    
    # L-005: Conditional Reasoning
    {"prompt": "If it's raining, the streets are wet. The streets are wet. Does that mean it rained?",
     "behavior": "L-005"},
    
    # L-006: Hidden Assumption Extraction
    {"prompt": "The argument is: 'This new drug should be approved because it passed clinical trials.' What assumption is being made?",
     "behavior": "L-006"},
]

# Load base model for comparison
print("Loading base model for comparison...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map="auto",
    torch_dtype=torch.bfloat16 if USE_4BIT else torch.float32,
)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

# Evaluate both models
print("\n" + "=" * 70)
print("EVALUATION: Base vs Fine-tuned")
print("=" * 70)

for i, eval_item in enumerate(EVAL_PROMPTS):
    prompt = eval_item["prompt"]
    behavior = eval_item["behavior"]
    
    print(f"\n--- {behavior} ---")
    print(f"Q: {prompt}")
    
    # Base model
    base_response = generate_response(base_model, base_tokenizer, prompt)
    print(f"\nBASE: {base_response[:300]}...")
    
    # Fine-tuned model
    ft_response = generate_response(model, tokenizer, prompt)
    print(f"\nFINE-TUNED: {ft_response[:300]}...")
    print()

In [ ]:
#@title 11. Upload to Hugging Face (Optional)
# Uncomment to upload the adapter to Hugging Face

# from huggingface_hub import HfApi
# 
# REPO_NAME = "your-username/lora-memory-rhetoric-logic-llama3.2-1b"
# 
# api = HfApi()
# api.create_repo(repo_id=REPO_NAME, exist_ok=True)
# api.upload_folder(
#     folder_path=adapter_path,
#     repo_id=REPO_NAME,
#     repo_type="model",
# )
# 
# print(f"Uploaded to: https://huggingface.co/{REPO_NAME}")

print("Training complete. Adapter saved locally.")
print("To upload, uncomment the code above and set your repo name.")